# Séance 4 — Prévoir les tâches en retard

**QUID : au démarrage d'une tâche, peut-on prévoir son risque de retard pour proposer de l'aide à l'équipe ?**

On dispose du délai prévu, de l'effort estimé, de la charge de l'équipe, du nombre de dépendances et de son expérience sur ce type de tâche. La réponse est `en_retard` : 0 = livrée à temps, 1 = livrée en retard.

**Données simulées pour le cours.** Nous inventons un historique de tâches avec des relations plausibles et du hasard. Les modèles cherchent à retrouver ces relations ; les scores ne prouvent aucune performance dans une entreprise réelle. Ce cas sert à comprendre l'apprentissage, pas à noter des salariés.

Le notebook est autonome et n'utilise aucune connexion Internet.

## 1. Construire un historique fictif

Dans cette simulation, un effort élevé, des dépendances et une équipe chargée peuvent allonger la durée. L'expérience peut la réduire. Des imprévus restent invisibles pour le modèle.

La durée réelle sert **uniquement à construire la réponse historique**. Elle ne sera jamais donnée aux modèles : au démarrage, on ne la connaît pas.

**À dire :** « Nous avons les fiches de tâches et nous savons, après coup, lesquelles ont fini en retard. »

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, roc_auc_score, ConfusionMatrixDisplay

rng = np.random.default_rng(42)
n_tasks = 1200
X = pd.DataFrame({
    "delai_prevu_jours": rng.integers(3, 21, n_tasks),
    "effort_estime_jours": rng.integers(1, 16, n_tasks),
    "charge_equipe_pct": rng.integers(30, 111, n_tasks),
    "dependances": rng.integers(0, 5, n_tasks),
    "experience_projets_similaires": rng.integers(0, 11, n_tasks),
})
duree_reelle = (
    X.effort_estime_jours * (0.8 + X.charge_equipe_pct / 150)
    + 1.1 * X.dependances - 0.35 * X.experience_projets_similaires
    + rng.normal(0, 2.5, n_tasks)
).clip(lower=1)
y = (duree_reelle > X.delai_prevu_jours).astype(int).rename("en_retard")
history = X.assign(en_retard=y)
display(history.head(8))
display(history.en_retard.value_counts().rename(index={0: "À temps", 1: "En retard"}))
print("Toutes les variables explicatives sont disponibles au démarrage de la tâche.")

## 2. Réserver un examen aux modèles

On entraîne sur 75 % des tâches et on teste sur les 25 % restantes. La séparation est stratifiée pour conserver les proportions des deux classes. La graine est fixée avant de regarder les résultats.

Les tâches de cette simulation sont indépendantes. Sur un historique réel, il faudrait notamment considérer les dates et éviter de mélanger des tâches d'un même projet entre train et test.

**À dire :** « Les modèles apprennent sur 900 tâches, puis passent le même examen sur 300 autres. »

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
assert set(X_train.index).isdisjoint(X_test.index)
display(pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()})
        .rename(index={0: "À temps", 1: "En retard"}))

## 3. Comparer trois modèles à une règle naïve

- **Référence :** annoncer toujours la classe la plus fréquente dans le train.
- **Arbre :** poser quelques questions sur les caractéristiques de la tâche.
- **Régression logistique :** combiner les caractéristiques pour estimer une probabilité de retard.
- **Forêt aléatoire :** faire voter plusieurs arbres.

Nous fixons les paramètres à l'avance. On ne les ajuste pas en regardant le test.

**Accuracy :** part des tâches bien classées. **Précision des alertes :** parmi les alertes, combien correspondent à un retard réel ? **Rappel des retards :** parmi les retards, combien sont détectés ?

In [ ]:
models = {
    "Référence naïve": DummyClassifier(strategy="most_frequent"),
    "Arbre simple": DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42),
    "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=5, random_state=42),
}
rows = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({"modèle": name, "bonnes réponses / 300": int((pred == y_test).sum()),
                 "accuracy (%)": 100 * accuracy_score(y_test, pred),
                 "précision alertes (%)": 100 * precision_score(y_test, pred, zero_division=0),
                 "retards détectés (%)": 100 * recall_score(y_test, pred, zero_division=0)})
results = pd.DataFrame(rows)
display(results.round(1))
ax = results.plot.barh(x="modèle", y="accuracy (%)", figsize=(8, 4), legend=False)
ax.set(xlim=(0, 100), xlabel="Tâches bien classées (%)", ylabel="", title="Même examen pour les quatre approches")
plt.tight_layout()
plt.show()

## 4. Comprendre les erreurs et les règles

Une fausse alerte peut mobiliser inutilement du temps d'aide. Un retard manqué peut empêcher l'équipe d'anticiper. Un score global ne décrit pas ce compromis.

La matrice de confusion de l'arbre montre les comptes exacts. Les règles dessinées ensuite permettent d'expliquer ses prédictions. `samples` compte les tâches d'entraînement et `value` donne les effectifs [à temps, en retard].

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, predictions["Arbre simple"], labels=[0, 1],
    display_labels=["À temps", "En retard"], ax=ax, colorbar=False)
ax.set_title("Arbre simple — tâches de test")
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(17, 7))
plot_tree(models["Arbre simple"], feature_names=list(X.columns),
          class_names=["À temps", "En retard"], filled=True, rounded=True,
          impurity=False, fontsize=9, ax=ax)
plt.tight_layout()
plt.show()

## 5. Une nouvelle tâche arrive

L'équipe a 7 jours pour une tâche estimée à 9 jours d'effort, avec 95 % de charge et trois dépendances. Que proposent les modèles ?

**À dire :** « Une alerte sert à ouvrir une discussion : peut-on donner plus de délai, réduire le périmètre ou apporter de l'aide ? »

Les probabilités affichées sont des estimations apprises dans notre simulation, pas des certitudes ni des probabilités validées dans une entreprise.

In [ ]:
new_task = pd.DataFrame([{
    "delai_prevu_jours": 7, "effort_estime_jours": 9,
    "charge_equipe_pct": 95, "dependances": 3,
    "experience_projets_similaires": 2,
}])
display(new_task)
for name, model in list(models.items())[1:]:
    probability = model.predict_proba(new_task)[0, 1]
    decision = "alerte retard" if model.predict(new_task)[0] else "pas d'alerte"
    print(f"{name} : {decision} (probabilité estimée : {probability:.0%})")

## Conclusion pour la classe

**La question :** peut-on repérer les tâches à risque au démarrage ?

**L'expérience :** des modèles apprennent sur un historique simulé et sont testés sur des tâches séparées. On compare leurs prédictions à une référence simple et on examine les erreurs.

**La limite :** nous avons créé un monde où les variables observées expliquent une bonne partie du retard. Une entreprise réelle peut fonctionner différemment ; il faudrait y collecter des données, vérifier leur disponibilité au bon moment et évaluer sur de nouveaux projets.

**Questions aux étudiants :** préfère-t-on rater moins de retards ou déclencher moins de fausses alertes ? Quelle action concrète prendrait-on après une alerte ?

Un modèle peut fonctionner sur un problème bien défini sans être parfait. Le score ne suffit pas : il faut aussi savoir ce qu'on fera de la prédiction.